# 07 Evaluate the End-to-End Pipeline

This notebook connects a selected NER prediction file to a selected trained RE
experiment. It automatically loads the RE marker design, context length, and
validation-selected positive threshold from Notebook 05.

The default next run uses the existing PubMedBERT NER predictions with the existing
BERT RE model. Change `RE_EXPERIMENT` after a new Notebook 05 experiment completes.
All runs retain the fixed report split and `seed=42`.


In [ ]:
from __future__ import annotations

import inspect
import json
import os
import sys
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from dotenv import load_dotenv
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing pyproject.toml")


def format_duration(seconds: float) -> str:
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

RUN_NAME = os.getenv("RADGRAPH_XL_RUN_NAME", "full_2300")
RUN_ROOT = PROJECT_ROOT / "outputs" / RUN_NAME
INTERIM_DIR = RUN_ROOT / "interim"
RESULTS_DIR = RUN_ROOT / "results"
PREDICTIONS_DIR = RUN_ROOT / "predictions"
ANALYSIS_DIR = RUN_ROOT / "analysis"
FIGURES_DIR = RUN_ROOT / "figures"
MODELS_DIR = RUN_ROOT / "models"

for path in [RESULTS_DIR, PREDICTIONS_DIR, ANALYSIS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# This already exists and provides the cheapest first test of whether the stronger
# NER backbone improves the complete pipeline.
NER_EXPERIMENT = "pubmedbert_sliding_window"

# Keep the existing RE baseline for the first PubMedBERT-NER run. After Notebook 05,
# set this to "bert_generic_neg5_unweighted" or another named RE experiment.
# RE_EXPERIMENT = "bert_base_uncased"
RE_EXPERIMENT = "bert_generic_neg5_unweighted"

# Use a threshold only when Notebook 05 has selected it on validation data. Otherwise
# the code safely falls back to the model's ordinary argmax prediction.
USE_TUNED_THRESHOLD = True

NER_PREDICTIONS_JSONL = PREDICTIONS_DIR / f"ner_{NER_EXPERIMENT}_test_predictions.jsonl"
RE_MODEL_DIR = MODELS_DIR / f"re_{RE_EXPERIMENT}" / "best_model"
RE_CONFIG_PATH = RESULTS_DIR / f"re_{RE_EXPERIMENT}_run_config.json"
RE_THRESHOLD_PATH = RESULTS_DIR / f"re_{RE_EXPERIMENT}_selected_threshold.json"
NER_JSONL = INTERIM_DIR / "ner_dataset.jsonl"
ENTITIES_CSV = INTERIM_DIR / "entities.csv"
RELATIONS_CSV = INTERIM_DIR / "relations.csv"

required_paths = [
    NER_PREDICTIONS_JSONL,
    RE_MODEL_DIR,
    RE_CONFIG_PATH,
    NER_JSONL,
    ENTITIES_CSV,
    RELATIONS_CSV,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, (
    "Missing end-to-end inputs:\n- " + "\n- ".join(missing_paths)
    + "\nRun notebooks 03, 05, and 06, or select completed experiment names."
)

re_config = json.loads(RE_CONFIG_PATH.read_text(encoding="utf-8"))
MARKER_DESIGN = re_config.get("marker_design", "generic")
CONTEXT_WINDOW = int(re_config.get("context_window", 64))
MAX_LENGTH = int(re_config.get("max_length", 256))
RELATION_DISTANCE_THRESHOLD = int(re_config.get("relation_distance_threshold", 32))

selected_positive_threshold = None
if USE_TUNED_THRESHOLD and RE_THRESHOLD_PATH.exists():
    threshold_config = json.loads(RE_THRESHOLD_PATH.read_text(encoding="utf-8"))
    selected_positive_threshold = float(
        threshold_config["selected_positive_threshold"]
    )

PREDICTION_BATCH_SIZE = 32
NO_RELATION_LABEL = "no_relation"
RANDOM_SEED = 42

RUN_SMOKE_TEST = False
MAX_DOCUMENTS = 3 if RUN_SMOKE_TEST else None
MAX_CANDIDATES = 2000 if RUN_SMOKE_TEST else None

print(
    {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "python_executable": sys.executable,
        "cuda_available": torch.cuda.is_available(),
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "ner_experiment": NER_EXPERIMENT,
        "re_experiment": RE_EXPERIMENT,
        "marker_design": MARKER_DESIGN,
        "context_window": CONTEXT_WINDOW,
        "max_length": MAX_LENGTH,
        "distance_threshold": RELATION_DISTANCE_THRESHOLD,
        "selected_positive_threshold": selected_positive_threshold,
        "random_seed": RANDOM_SEED,
        "run_smoke_test": RUN_SMOKE_TEST,
    }
)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select the project's .venv as the notebook kernel.")


In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as handle:
        total = sum(1 for line in handle if line.strip())
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in tqdm(handle, total=total, desc=f"Loading {path.name}", unit="row"):
            if line.strip():
                rows.append(json.loads(line))
    return rows


ner_rows = load_jsonl(NER_JSONL)
prediction_rows = load_jsonl(NER_PREDICTIONS_JSONL)

if MAX_DOCUMENTS is not None:
    prediction_rows = prediction_rows[:MAX_DOCUMENTS]

test_doc_ids = {row["doc_id"] for row in prediction_rows}
tokens_by_doc = {
    row["doc_id"]: row["tokens"]
    for row in ner_rows
    if row["doc_id"] in test_doc_ids
}
predicted_entities_by_doc = {
    row["doc_id"]: row["predicted_entities"]
    for row in prediction_rows
}

entities = pd.read_csv(ENTITIES_CSV)
relations = pd.read_csv(RELATIONS_CSV)
entities = entities[
    (entities["split"] == "test") & (entities["doc_id"].isin(test_doc_ids))
].copy()
relations = relations[
    (relations["split"] == "test") & (relations["doc_id"].isin(test_doc_ids))
].copy()

# Notebook 03 excludes entity pairs with more than one gold relation label because the
# RE classifier predicts one class per ordered pair. Apply the same rule here.
pair_columns = [
    "doc_id",
    "head_start",
    "head_end",
    "tail_start",
    "tail_end",
]
relation_label_counts = relations.groupby(pair_columns)["safe_label"].transform("nunique")
ambiguous_relation_rows_removed = int((relation_label_counts > 1).sum())
relations = relations[relation_label_counts == 1].copy()

assert set(test_doc_ids) <= set(tokens_by_doc), (
    "Some NER prediction documents are missing from ner_dataset.jsonl."
)

print(
    {
        "test_documents": len(test_doc_ids),
        "predicted_entities": sum(
            len(items) for items in predicted_entities_by_doc.values()
        ),
        "gold_entities": len(entities),
        "gold_relations_after_ambiguity_filter": len(relations),
        "ambiguous_relation_rows_removed": ambiguous_relation_rows_removed,
    }
)


In [ ]:
def span_distance(a_start: int, a_end: int, b_start: int, b_end: int) -> int:
    if a_end < b_start:
        return b_start - a_end
    if b_end < a_start:
        return a_start - b_end
    return 0


def entity_marker_type(label: str) -> str:
    normalised = str(label).strip().lower()
    if normalised.startswith("anatomy"):
        return "ANAT"
    if normalised.startswith("observation"):
        return "OBS"
    raise ValueError(f"Unsupported entity label for typed markers: {label}")


def marker_tokens(role: str, entity_label: str) -> tuple[str, str]:
    if MARKER_DESIGN == "generic":
        return f"[{role}]", f"[/{role}]"
    entity_type = entity_marker_type(entity_label)
    return f"[{role}_{entity_type}]", f"[/{role}_{entity_type}]"


def build_marked_tokens(
    tokens: list[str],
    head_start: int,
    head_end: int,
    tail_start: int,
    tail_end: int,
    head_label: str,
    tail_label: str,
    context_window: int,
) -> list[str]:
    left = max(0, min(head_start, tail_start) - context_window)
    right = min(len(tokens), max(head_end, tail_end) + context_window + 1)
    head_open, head_close = marker_tokens("HEAD", head_label)
    tail_open, tail_close = marker_tokens("TAIL", tail_label)
    marked_tokens = []
    for index in range(left, right):
        if index == head_start:
            marked_tokens.append(head_open)
        if index == tail_start:
            marked_tokens.append(tail_open)
        marked_tokens.append(tokens[index])
        if index == head_end:
            marked_tokens.append(head_close)
        if index == tail_end:
            marked_tokens.append(tail_close)
    return marked_tokens


candidate_rows = []
candidate_tokens = []

for doc_id in tqdm(
    sorted(test_doc_ids),
    desc="Generating predicted-entity pairs",
    unit="report",
):
    tokens = tokens_by_doc[doc_id]
    unique_entities = {
        (int(entity["start"]), int(entity["end"]), entity["label"])
        for entity in predicted_entities_by_doc[doc_id]
    }
    doc_entities = [
        {
            "entity_id": f"pred_{index}",
            "start": start,
            "end": end,
            "label": label,
        }
        for index, (start, end, label) in enumerate(sorted(unique_entities))
        if 0 <= start <= end < len(tokens)
    ]

    for head in doc_entities:
        for tail in doc_entities:
            if head["entity_id"] == tail["entity_id"]:
                continue
            distance = span_distance(
                head["start"],
                head["end"],
                tail["start"],
                tail["end"],
            )
            if distance > RELATION_DISTANCE_THRESHOLD:
                continue
            candidate_rows.append(
                {
                    "doc_id": doc_id,
                    "head_entity_id": head["entity_id"],
                    "tail_entity_id": tail["entity_id"],
                    "head_start": head["start"],
                    "head_end": head["end"],
                    "tail_start": tail["start"],
                    "tail_end": tail["end"],
                    "head_label": head["label"],
                    "tail_label": tail["label"],
                    "distance": distance,
                }
            )
            candidate_tokens.append(
                build_marked_tokens(
                    tokens=tokens,
                    head_start=head["start"],
                    head_end=head["end"],
                    tail_start=tail["start"],
                    tail_end=tail["end"],
                    head_label=head["label"],
                    tail_label=tail["label"],
                    context_window=CONTEXT_WINDOW,
                )
            )

if MAX_CANDIDATES is not None and len(candidate_rows) > MAX_CANDIDATES:
    candidate_rows = candidate_rows[:MAX_CANDIDATES]
    candidate_tokens = candidate_tokens[:MAX_CANDIDATES]

candidate_frame = pd.DataFrame(candidate_rows)
assert len(candidate_frame) == len(candidate_tokens)
assert not candidate_frame.empty, "No predicted-entity relation candidates were generated."

print(
    {
        "candidate_pairs": len(candidate_frame),
        "median_distance": float(candidate_frame["distance"].median()),
        "maximum_distance": int(candidate_frame["distance"].max()),
    }
)


In [ ]:
load_started = time.perf_counter()
print("Loading relation tokenizer and model from", RE_MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(RE_MODEL_DIR, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(RE_MODEL_DIR)
print("Model loaded in", format_duration(time.perf_counter() - load_started))

relation_id_to_label = {
    int(index): label
    for index, label in model.config.id2label.items()
}
relation_label_to_id = {
    label: index
    for index, label in relation_id_to_label.items()
}
assert NO_RELATION_LABEL in relation_label_to_id, "RE model has no no_relation label."

configured_special_tokens = set(re_config.get("special_tokens", []))
marker_token_ids = {
    tokenizer.convert_tokens_to_ids(token)
    for token in configured_special_tokens
}
marker_token_ids.discard(tokenizer.unk_token_id)

inference_dataset = Dataset.from_list(
    [{"tokens": tokens} for tokens in candidate_tokens]
)


def tokenize_batch(batch: dict) -> dict:
    tokenized = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    if marker_token_ids:
        for row_index, input_ids in enumerate(tokenized["input_ids"]):
            marker_count = sum(
                int(token_id in marker_token_ids)
                for token_id in input_ids
            )
            if marker_count != 4:
                raise ValueError(
                    "End-to-end tokenisation removed an entity marker. "
                    f"Batch row {row_index} retained {marker_count}/4 markers."
                )
    return tokenized


tokenized_dataset = inference_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=inference_dataset.column_names,
    desc="Tokenising end-to-end RE candidates",
)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

inference_args = TrainingArguments(
    output_dir=str(MODELS_DIR / "_end_to_end_inference"),
    per_device_eval_batch_size=PREDICTION_BATCH_SIZE,
    report_to=[],
    disable_tqdm=False,
    dataloader_num_workers=0,
)
trainer_kwargs = {
    "model": model,
    "args": inference_args,
    "data_collator": data_collator,
}
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer
trainer = Trainer(**trainer_kwargs)

print("Starting relation inference")
prediction_started = time.perf_counter()
prediction_output = trainer.predict(tokenized_dataset)
logits = prediction_output.predictions
probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()

argmax_ids = np.argmax(probabilities, axis=-1)
predicted_ids = argmax_ids.copy()

positive_label_ids = np.array(
    [
        index
        for index, label in relation_id_to_label.items()
        if label != NO_RELATION_LABEL
    ],
    dtype=int,
)
if selected_positive_threshold is not None:
    positive_probabilities = probabilities[:, positive_label_ids]
    best_positive_offsets = positive_probabilities.argmax(axis=1)
    best_positive_ids = positive_label_ids[best_positive_offsets]
    best_positive_scores = positive_probabilities.max(axis=1)
    predicted_ids = np.where(
        best_positive_scores >= selected_positive_threshold,
        best_positive_ids,
        relation_label_to_id[NO_RELATION_LABEL],
    )

candidate_frame["argmax_label"] = [
    relation_id_to_label[int(index)]
    for index in argmax_ids
]
candidate_frame["predicted_label"] = [
    relation_id_to_label[int(index)]
    for index in predicted_ids
]
candidate_frame["prediction_confidence"] = probabilities.max(axis=-1)
candidate_frame["selected_positive_threshold"] = selected_positive_threshold
for label_id, label in relation_id_to_label.items():
    candidate_frame[f"probability_{label}"] = probabilities[:, int(label_id)]

predicted_relations = candidate_frame[
    candidate_frame["predicted_label"] != NO_RELATION_LABEL
].copy()

prediction_path = (
    PREDICTIONS_DIR
    / f"end_to_end_{NER_EXPERIMENT}_{RE_EXPERIMENT}_relations.csv"
)
predicted_relations.to_csv(prediction_path, index=False)

print(
    "Relation inference finished in",
    format_duration(time.perf_counter() - prediction_started),
)
print(
    {
        "decision_rule": (
            f"validation threshold {selected_positive_threshold}"
            if selected_positive_threshold is not None
            else "argmax"
        ),
        "predicted_positive_relations": len(predicted_relations),
        "saved": str(prediction_path),
    }
)


In [ ]:
def set_metrics(gold: set[tuple], predicted: set[tuple]) -> dict:
    true_positives = len(gold & predicted)
    false_positives = len(predicted - gold)
    false_negatives = len(gold - predicted)
    precision = true_positives / (true_positives + false_positives) if true_positives + false_positives else 0.0
    recall = true_positives / (true_positives + false_negatives) if true_positives + false_negatives else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_positives": true_positives,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "gold": len(gold),
        "predicted": len(predicted),
    }


gold_entity_set = {
    (row.doc_id, int(row.start), int(row.end), row.safe_label)
    for row in entities.itertuples(index=False)
}
predicted_entity_set = {
    (doc_id, int(entity["start"]), int(entity["end"]), entity["label"])
    for doc_id, doc_entities in predicted_entities_by_doc.items()
    for entity in doc_entities
}

gold_span_relation_set = {
    (
        row.doc_id,
        int(row.head_start),
        int(row.head_end),
        int(row.tail_start),
        int(row.tail_end),
        row.safe_label,
    )
    for row in relations.itertuples(index=False)
}
predicted_span_relation_set = {
    (
        row.doc_id,
        int(row.head_start),
        int(row.head_end),
        int(row.tail_start),
        int(row.tail_end),
        row.predicted_label,
    )
    for row in predicted_relations.itertuples(index=False)
}

gold_entity_label_lookup = {
    (row.doc_id, int(row.start), int(row.end)): row.safe_label
    for row in entities.itertuples(index=False)
}
gold_graph_relation_set = {
    (
        row.doc_id,
        int(row.head_start),
        int(row.head_end),
        gold_entity_label_lookup.get((row.doc_id, int(row.head_start), int(row.head_end)), "<missing>"),
        int(row.tail_start),
        int(row.tail_end),
        gold_entity_label_lookup.get((row.doc_id, int(row.tail_start), int(row.tail_end)), "<missing>"),
        row.safe_label,
    )
    for row in relations.itertuples(index=False)
}
predicted_graph_relation_set = {
    (
        row.doc_id,
        int(row.head_start),
        int(row.head_end),
        row.head_label,
        int(row.tail_start),
        int(row.tail_end),
        row.tail_label,
        row.predicted_label,
    )
    for row in predicted_relations.itertuples(index=False)
}

metrics = {
    "ner_experiment": NER_EXPERIMENT,
    "re_experiment": RE_EXPERIMENT,
    "relation_distance_threshold": RELATION_DISTANCE_THRESHOLD,
    "marker_design": MARKER_DESIGN,
    "selected_positive_threshold": selected_positive_threshold,
    "ambiguous_relation_rows_removed": ambiguous_relation_rows_removed,
    "random_seed": RANDOM_SEED,
    "marker_design": MARKER_DESIGN,
    "selected_positive_threshold": selected_positive_threshold,
    "ambiguous_relation_rows_removed": ambiguous_relation_rows_removed,
    "random_seed": RANDOM_SEED,
    "candidate_pairs": len(candidate_frame),
    "ner_strict_entity": set_metrics(gold_entity_set, predicted_entity_set),
    "end_to_end_relation_span_and_label": set_metrics(gold_span_relation_set, predicted_span_relation_set),
    "end_to_end_graph_entity_and_relation_labels": set_metrics(gold_graph_relation_set, predicted_graph_relation_set),
}

relation_labels = sorted(set(relations["safe_label"]) | set(predicted_relations["predicted_label"]))
per_relation_rows = []
for relation_label in relation_labels:
    gold_subset = {item for item in gold_span_relation_set if item[-1] == relation_label}
    predicted_subset = {item for item in predicted_span_relation_set if item[-1] == relation_label}
    row = {"relation_label": relation_label, **set_metrics(gold_subset, predicted_subset)}
    per_relation_rows.append(row)

per_relation = pd.DataFrame(per_relation_rows)
per_relation_path = RESULTS_DIR / f"end_to_end_{NER_EXPERIMENT}_{RE_EXPERIMENT}_per_relation.csv"
per_relation.to_csv(per_relation_path, index=False)

metrics_path = RESULTS_DIR / f"end_to_end_{NER_EXPERIMENT}_{RE_EXPERIMENT}_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print(json.dumps(metrics, indent=2))
display(per_relation)
print("Saved metrics:", metrics_path)
print("Saved per-relation table:", per_relation_path)

In [ ]:
false_positive_keys = predicted_span_relation_set - gold_span_relation_set
false_negative_keys = gold_span_relation_set - predicted_span_relation_set

error_columns = [
    "doc_id",
    "head_start",
    "head_end",
    "tail_start",
    "tail_end",
    "relation_label",
]

false_positive_frame = pd.DataFrame(list(false_positive_keys), columns=error_columns)
false_negative_frame = pd.DataFrame(list(false_negative_keys), columns=error_columns)

false_positive_path = ANALYSIS_DIR / f"end_to_end_{NER_EXPERIMENT}_{RE_EXPERIMENT}_false_positives.csv"
false_negative_path = ANALYSIS_DIR / f"end_to_end_{NER_EXPERIMENT}_{RE_EXPERIMENT}_false_negatives.csv"
false_positive_frame.to_csv(false_positive_path, index=False)
false_negative_frame.to_csv(false_negative_path, index=False)

doc_to_dataset = entities.drop_duplicates("doc_id").set_index("doc_id")["dataset"].to_dict()
dataset_rows = []
for dataset_name in sorted(set(doc_to_dataset.values())):
    doc_ids = {doc_id for doc_id, name in doc_to_dataset.items() if name == dataset_name}
    gold_subset = {item for item in gold_span_relation_set if item[0] in doc_ids}
    predicted_subset = {item for item in predicted_span_relation_set if item[0] in doc_ids}
    dataset_rows.append({"dataset": dataset_name, **set_metrics(gold_subset, predicted_subset)})

dataset_metrics = pd.DataFrame(dataset_rows)
dataset_metrics_path = RESULTS_DIR / f"end_to_end_{NER_EXPERIMENT}_{RE_EXPERIMENT}_by_dataset.csv"
dataset_metrics.to_csv(dataset_metrics_path, index=False)

display(dataset_metrics)
print("Saved false positives:", false_positive_path)
print("Saved false negatives:", false_negative_path)
print("Saved dataset metrics:", dataset_metrics_path)

In [ ]:
stage_rows = [
    {
        "stage": "NER strict entity",
        **metrics["ner_strict_entity"],
    },
    {
        "stage": "End-to-end relation (span + relation)",
        **metrics["end_to_end_relation_span_and_label"],
    },
    {
        "stage": "End-to-end graph (entity + relation labels)",
        **metrics["end_to_end_graph_entity_and_relation_labels"],
    },
]
stage_comparison = pd.DataFrame(stage_rows)
stage_path = RESULTS_DIR / f"end_to_end_{NER_EXPERIMENT}_{RE_EXPERIMENT}_stage_comparison.csv"
stage_comparison.to_csv(stage_path, index=False)
display(stage_comparison[["stage", "precision", "recall", "f1", "gold", "predicted"]])

fig, ax = plt.subplots(figsize=(9, 4.5))
colors = ["#3A7D44", "#2F6690", "#8F5D2E"]
ax.barh(stage_comparison["stage"], stage_comparison["f1"], color=colors)
ax.set_xlim(0, 1)
ax.set_xlabel("Strict F1")
ax.set_ylabel("")
ax.set_title("End-to-End Pipeline Performance")
ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
figure_path = FIGURES_DIR / f"end_to_end_{NER_EXPERIMENT}_{RE_EXPERIMENT}_pipeline_f1.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()

summary_lines = [
    "# End-to-End Pipeline Summary",
    "",
    f"NER experiment: `{NER_EXPERIMENT}`",
    "",
    f"RE experiment: `{RE_EXPERIMENT}`",
    "",
    f"Relation distance threshold: `{RELATION_DISTANCE_THRESHOLD}` tokens",
    "",
    f"Marker design: `{MARKER_DESIGN}`",
    "",
    f"Positive threshold: `{selected_positive_threshold if selected_positive_threshold is not None else 'argmax'}`",
    "",
    f"Marker design: `{MARKER_DESIGN}`",
    "",
    f"Positive threshold: `{selected_positive_threshold if selected_positive_threshold is not None else 'argmax'}`",
    "",
    "## Stage Metrics",
    "",
    stage_comparison[["stage", "precision", "recall", "f1", "gold", "predicted"]].to_markdown(index=False),
    "",
    "## Per-Relation Metrics",
    "",
    per_relation.to_markdown(index=False),
    "",
    "Gold-entity RE and end-to-end RE should be discussed separately because the latter includes NER error propagation.",
]
summary_path = RESULTS_DIR / f"end_to_end_{NER_EXPERIMENT}_{RE_EXPERIMENT}_summary.md"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")

print("Saved stage comparison:", stage_path)
print("Saved figure:", figure_path)
print("Saved summary:", summary_path)